<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/Deep_Learning_Project/blob/main/Project_02_Diabetes_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Diabetes prediction model use ANN

In [ ]:
# package import
import zipfile
import os

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.metrics import Accuracy, Precision, Recall


Dataset load from github

In [ ]:
!wget https://raw.githubusercontent.com/mdmostafizurrahman6351/Maching_learning/refs/heads/main/Deep_Learning/Dataset/diabetes_prediction_dataset.zip
zip_path = 'diabetes_prediction_dataset.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall('data')

os.listdir('data')


In [ ]:
# data read
df = pd.read_csv('/content/data/diabetes_prediction_dataset.csv')
input_cols = df.drop('diabetes', axis=1).columns.tolist()
print(df.shape)
df.head()
input_cols


In [ ]:
# gender and smoking_history columns lower case
df['gender'] = df['gender'].str.lower()
df['smoking_history'] = df['smoking_history'].str.lower()
df.head()

Data preprocessing

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.value_counts('diabetes')


In [ ]:

encoder_gender = OneHotEncoder(drop='first', handle_unknown='ignore')
encoder_smoking_history = OneHotEncoder(drop='first', handle_unknown='ignore')

encoded_gen = encoder_gender.fit_transform(df[['gender']])
encoded_smoking_history = encoder_smoking_history.fit_transform(df[['smoking_history']])

encoded_gen = pd.DataFrame(encoded_gen.toarray(), columns=encoder_gender.get_feature_names_out(['gender']))
encoded_smoking_history = pd.DataFrame(encoded_smoking_history.toarray(), columns=encoder_smoking_history.get_feature_names_out(['smoking_history']))

df = pd.concat([df.drop(['gender', 'smoking_history'], axis=1), encoded_gen, encoded_smoking_history], axis=1)
print(df.shape)
df.head()

In [ ]:
# encoding model save
joblib.dump(encoder_gender, 'encoder_gender.joblib')
joblib.dump(encoder_smoking_history, 'encoder_smoking_history.joblib')

In [ ]:
sns.countplot(x='diabetes', data=df)

In [ ]:
df.hist(figsize=(12,10))
plt.show()

In [ ]:
plt.figure(figsize=(12,10))

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
sns.boxplot(x='diabetes', y='blood_glucose_level', data=df)
plt.show()

outlayer detection and remove

In [ ]:
x = df.drop('diabetes', axis=1)
y = df['diabetes']

In [ ]:
# emblance data heandling
smote = SMOTE(random_state=0)
x_smote, y_smote = smote.fit_resample(x,y)
df = pd.concat([x_smote, y_smote], axis=1)
print(df.shape)
df.head()

In [ ]:
df.value_counts('diabetes')

In [ ]:
 # Outlier detection code (commented out due to incorrect logic)
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

outlayer_detection = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR)))
outlayer_detection.sum()

In [ ]:
#Outlier removal code (commented out due to incorrect logic)
df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

Feature and tergate

In [ ]:
x = df.drop('diabetes', axis=1)
y = df['diabetes']

train test split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)


Scalling

In [ ]:
scaller = StandardScaler()
x_train = scaller.fit_transform(x_train)
x_test = scaller.transform(x_test)

In [ ]:
# scalling model save
joblib.dump(scaller, 'scaller.joblib')

ANN model build

In [ ]:
# model create
model = Sequential([
    # fast layer
    layers.Dense(128, activation='relu', input_shape=(x_train.shape[1],)),
    # hidden layer
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    #output layer
    layers.Dense(1, activation='sigmoid')
])
model.summary()



In [ ]:
# model compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# model fit
hostory = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=250,
    validation_split=0.2
)


In [ ]:
#model evaluate
loss, accuracy = model.evaluate(x_test, y_test)
y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print('confusion matrix', confusion_matrix(y_test, y_pred))
print('classification report', classification_report(y_test, y_pred))

In [ ]:
#model evaluate
loss, accuracy = model.evaluate(x_test, y_test)
y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print('confusion matrix', confusion_matrix(y_test, y_pred))
print('classification report', classification_report(y_test, y_pred))

# Create a Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix Heatmap')
plt.show()

model save

In [ ]:
# save model
model.save('Diabetes predction model.h5')

predction from new data

In [ ]:
class diabetes():

  def __init__(self):
    self.input_cols = ['gender','age','hypertension','heart_disease','smoking_history','bmi','HbA1c_level','blood_glucose_level']
    import tensorflow as tf
    self.model2 = tf.keras.models.load_model('/content/Diabetes predction model.h5')
    self.encoder_gender = joblib.load('/content/encoder_gender.joblib')
    self.encoder_smoking_history = joblib.load('/content/encoder_smoking_history.joblib')
    self.scaller = joblib.load('/content/scaller.joblib')
  def input_data(self):
    new_data = pd.DataFrame(columns=self.input_cols)
    print("Please provide the following information:")
    for col in self.input_cols:
      new_data[col] = [input(f'Enter {col}: ')]
    return new_data

  def encoding(self, new_data):
    # Convert gender and smoking_history to lowercase, similar to initial preprocessing
    new_data['gender'] = new_data['gender'].str.lower()
    new_data['smoking_history'] = new_data['smoking_history'].str.lower()

    encoded_gen = self.encoder_gender.transform(new_data[['gender']])
    encoded_smoking_history = self.encoder_smoking_history.transform(new_data[['smoking_history']])

    encoded_gen = pd.DataFrame(encoded_gen.toarray(), columns=self.encoder_gender.get_feature_names_out(['gender']))
    encoded_smoking_history = pd.DataFrame(encoded_smoking_history.toarray(), columns=self.encoder_smoking_history.get_feature_names_out(['smoking_history']))

    enc_data = pd.concat([new_data.drop(['gender', 'smoking_history'], axis=1), encoded_gen, encoded_smoking_history], axis=1)

    return enc_data

  def scalling(self, enc_data):
    # Convert numerical columns from string to numeric type before scaling
    numeric_cols_from_input = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']
    for col in numeric_cols_from_input:
      if col in enc_data.columns: # Check if column exists in enc_data
        enc_data[col] = pd.to_numeric(enc_data[col])

    scal_data = self.scaller.transform(enc_data)
    return scal_data

  def prediction(self, scal_data):
    diabetes_lavel = self.model2.predict(scal_data)

    return diabetes_lavel

  def output(self, diabetes_lavel):
    if diabetes_lavel[0][0] > 0.5: # Assuming a threshold of 0.5 for binary classification
      print('Diabetes')
    else:
      print('No diabetes')


  def check(self):
    df = self.input_data()
    df = self.encoding(df)
    df = self.scalling(df)
    lavel = self.prediction(df)
    self.output(lavel)
    return lavel


In [ ]:
# model use
predictor = diabetes()
predictor.check()
